## Fine-tune the same model using LORA

In [ ]:
import os
import json
from dataclasses import dataclass
from typing import List, Dict, Optional

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    set_seed,
)
from peft import LoraConfig, get_peft_model

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B"

TRAIN_FILE = "train.jsonl"
VAL_FILE = "val.jsonl"  
cache_dir = "./cache"
LORA_OUTPUT_DIR = "./qwen2.5-1.5b-lora-sft"
MAX_LENGTH = 512
SEED = 42

NUM_EPOCHS = 1
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 16
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.0
WARMUP_RATIO = 0.03
LOGGING_STEPS = 10
EVAL_STEPS = 100
SAVE_STEPS = 100
SAVE_TOTAL_LIMIT = 2

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP16 = torch.cuda.is_available() and not USE_BF16

os.makedirs(LORA_OUTPUT_DIR, exist_ok=True)
set_seed(SEED)

print("bf16:", USE_BF16)
print("fp16:", USE_FP16)

In [ ]:
PROMPT_WITH_INPUT = """### Instruction:
{instruction}

### Input:
{input}

### Response:
"""

PROMPT_NO_INPUT = """### Instruction:
{instruction}

### Response:
"""


def format_prompt(example: Dict[str, str]) -> str:
    instruction = example["instruction"].strip()
    input_text = example.get("input", "")
    input_text = "" if input_text is None else input_text.strip()

    if input_text:
        return PROMPT_WITH_INPUT.format(
            instruction=instruction,
            input=input_text
        )
    else:
        return PROMPT_NO_INPUT.format(
            instruction=instruction
        )

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    cache_dir=cache_dir,
    use_fast=False,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("pad_token:", tokenizer.pad_token)
print("eos_token:", tokenizer.eos_token)

In [ ]:
def tokenize_example(example: Dict[str, str], max_length: int = MAX_LENGTH):
    prompt = format_prompt(example)
    response = example["output"].strip() + tokenizer.eos_token

    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(prompt + response, add_special_tokens=False)["input_ids"]

    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]

    full_ids = full_ids[:max_length]
    labels = labels[:max_length]
    attention_mask = [1] * len(full_ids)

    return {
        "input_ids": full_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

In [ ]:
data_files = {"train": TRAIN_FILE}
if os.path.exists(VAL_FILE):
    data_files["validation"] = VAL_FILE

raw_datasets = load_dataset("json", data_files=data_files)
raw_datasets

In [ ]:
tokenized_train = raw_datasets["train"].map(
    tokenize_example,
    remove_columns=raw_datasets["train"].column_names,
)

tokenized_val = None
if "validation" in raw_datasets:
    tokenized_val = raw_datasets["validation"].map(
        tokenize_example,
        remove_columns=raw_datasets["validation"].column_names,
    )

print(tokenized_train[0])
if tokenized_val is not None:
    print("validation size:", len(tokenized_val))

In [ ]:
@dataclass
class DataCollatorForCausalLMWithPadding:
    tokenizer: AutoTokenizer
    pad_to_multiple_of: Optional[int] = 8

    def __call__(self, features: List[Dict[str, List[int]]]):
        input_ids = [f["input_ids"] for f in features]
        attention_mask = [f["attention_mask"] for f in features]
        labels = [f["labels"] for f in features]

        batch_input_ids = self._pad(input_ids, self.tokenizer.pad_token_id)
        batch_attention_mask = self._pad(attention_mask, 0)
        batch_labels = self._pad(labels, -100)

        return {
            "input_ids": torch.tensor(batch_input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(batch_attention_mask, dtype=torch.long),
            "labels": torch.tensor(batch_labels, dtype=torch.long),
        }

    def _pad(self, sequences: List[List[int]], pad_value: int):
        max_len = max(len(x) for x in sequences)
        if self.pad_to_multiple_of is not None and max_len % self.pad_to_multiple_of != 0:
            max_len = ((max_len // self.pad_to_multiple_of) + 1) * self.pad_to_multiple_of
        return [seq + [pad_value] * (max_len - len(seq)) for seq in sequences]


data_collator = DataCollatorForCausalLMWithPadding(tokenizer=tokenizer)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    cache_dir=cache_dir,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16 if USE_BF16 else (torch.float16 if USE_FP16 else torch.float32),
)

model.config.use_cache = False
model.gradient_checkpointing_enable()

print("Base model loaded.")

### Attach Lora

In [ ]:
target_modules = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj"
]

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=target_modules,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
training_args = TrainingArguments(
    output_dir=LORA_OUTPUT_DIR,
    overwrite_output_dir=True,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    logging_steps=LOGGING_STEPS,
    eval_steps=EVAL_STEPS if tokenized_val is not None else None,
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    evaluation_strategy="steps" if tokenized_val is not None else "no",
    save_strategy="steps",
    bf16=USE_BF16,
    fp16=USE_FP16,
    gradient_checkpointing=True,
    lr_scheduler_type="cosine",
    report_to="none",
    remove_unused_columns=False,
    label_names=["labels"],
    dataloader_pin_memory=True,
    optim="adamw_torch",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer.train()

In [ ]:
trainer.model.save_pretrained(LORA_OUTPUT_DIR)
tokenizer.save_pretrained(LORA_OUTPUT_DIR)

with open(os.path.join(LORA_OUTPUT_DIR, "train_config.json"), "w", encoding="utf-8") as f:
    json.dump(
        {
            "model_name": MODEL_NAME,
            "train_file": TRAIN_FILE,
            "val_file": VAL_FILE,
            "output_dir": LORA_OUTPUT_DIR,
            "max_length": MAX_LENGTH,
            "num_epochs": NUM_EPOCHS,
            "train_batch_size": TRAIN_BATCH_SIZE,
            "eval_batch_size": EVAL_BATCH_SIZE,
            "grad_accum_steps": GRAD_ACCUM_STEPS,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "warmup_ratio": WARMUP_RATIO,
            "seed": SEED,
            "lora_r": LORA_R,
            "lora_alpha": LORA_ALPHA,
            "lora_dropout": LORA_DROPOUT,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

print("LoRA adapter saved to:", LORA_OUTPUT_DIR)

In [ ]:
from peft import PeftModel

infer_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    cache_dir=cache_dir,
    use_fast=False,
    trust_remote_code=True
)
if infer_tokenizer.pad_token is None:
    infer_tokenizer.pad_token = infer_tokenizer.eos_token

base_model_for_infer = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    cache_dir=cache_dir,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16 if USE_BF16 else (torch.float16 if USE_FP16 else torch.float32),
)

infer_model = PeftModel.from_pretrained(base_model_for_infer, LORA_OUTPUT_DIR)

if torch.cuda.is_available():
    infer_model = infer_model.cuda()

infer_model.eval()
print("LoRA inference model loaded.")

In [ ]:
def build_prompt(instruction: str, input_text: str = ""):
    instruction = instruction.strip()
    input_text = (input_text or "").strip()

    if input_text:
        return PROMPT_WITH_INPUT.format(
            instruction=instruction,
            input=input_text
        )
    else:
        return PROMPT_NO_INPUT.format(
            instruction=instruction
        )


def generate_response(
    model,
    tokenizer,
    instruction: str,
    input_text: str = "",
    max_new_tokens: int = 128,
):
    prompt = build_prompt(instruction, input_text)

    inputs = tokenizer(prompt, return_tensors="pt")
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            top_p=1.0,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if full_text.startswith(prompt):
        return full_text[len(prompt):].strip()
    return full_text.strip()

In [ ]:
tests = [
    {
        "instruction": "Rewrite the sentence in a polite tone.",
        "input": "Send me the report today."
    },
    {
        "instruction": "Rewrite the sentence in a formal academic style.",
        "input": "I can't come to class tomorrow."
    },
    {
        "instruction": "Simplify the sentence.",
        "input": "The committee reached a unanimous decision."
    },
    {
        "instruction": "Summarize the paragraph into exactly 3 bullet points.",
        "input": "Large language models are trained on large amounts of text data. They can generate fluent responses, but they may also produce incorrect information. Fine-tuning can improve their behavior on specific tasks."
    }
]

for i, ex in enumerate(tests, 1):
    print("=" * 80)
    print(f"Test {i}")
    print("Instruction:", ex["instruction"])
    print("Input:", ex["input"])
    print("-" * 80)
    out = generate_response(
        infer_model,
        infer_tokenizer,
        ex["instruction"],
        ex["input"],
        max_new_tokens=128
    )
    print(out)
    print()

In [ ]:
base_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    cache_dir=cache_dir,
    use_fast=False,
    trust_remote_code=True
)
if base_tokenizer.pad_token is None:
    base_tokenizer.pad_token = base_tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    cache_dir=cache_dir,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16 if USE_BF16 else (torch.float16 if USE_FP16 else torch.float32),
)
if torch.cuda.is_available():
    base_model = base_model.cuda()
base_model.eval()

print("Base model loaded.")

def compare_base_vs_ft(instruction, input_text, max_new_tokens=128):
    base_out = generate_response(
        base_model, base_tokenizer, instruction, input_text, max_new_tokens=max_new_tokens
    )
    ft_out = generate_response(
        infer_model, infer_tokenizer, instruction, input_text, max_new_tokens=max_new_tokens
    )

    print("=" * 100)
    print("INSTRUCTION:", instruction)
    print("INPUT:", input_text)
    print("-" * 100)
    print("BASE MODEL:")
    print(base_out)
    print("-" * 100)
    print("FULL-FT MODEL:")
    print(ft_out)
    print("=" * 100)

compare_base_vs_ft(
    "Rewrite the sentence in a polite tone.",
    "Send me the report today."
)

compare_base_vs_ft(
    "Summarize the paragraph into exactly 3 bullet points.",
    "Large language models are trained on large amounts of text data. They can generate fluent responses, but they may also produce incorrect information. Fine-tuning can improve their behavior on specific tasks."
)

### Tasks

- Train the models for longer, compare the results.
- Add task to the training dataset, and evaluate the fine-tuned model